# Пункт 5. Обучение LoRA на восьми примерах

Модель `Qwen/Qwen3-4B-Instruct-2507` обучается на восьми зафиксированных
train-примерах корпуса v3. Они представляют восемь категорий,
четыре опасных и четыре обычных вопроса.

Откройте новую сессию **Colab T4 GPU** и выполняйте ячейки сверху вниз.
Выбраны LR 5e-5, LoRA r16 / alpha32, 24 шага оптимизатора,
batch 1 и накопление четырёх микробатчей. Это 12 проходов по восьми примерам.

Ноутбук сохранит ответы до обучения, после него и после загрузки адаптера,
loss на восьми train и всех 16 validation-примерах, журнал шагов и веса.
Для контроля используется greedy с лимитом 768 токенов.
Содержательное сравнение на development выполняется в следующем пилоте.

[Подробная инструкция](https://github.com/maria-igntv/lm-hw/blob/main/docs/sft_v3_tiny_guide.md).


In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/lm-hw-v3")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/maria-igntv/lm-hw.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
settings = json.loads((ROOT / "data/sft_v3/tiny_config.json").read_text(encoding="utf-8"))
selection = json.loads((ROOT / settings["selection_path"]).read_text(encoding="utf-8"))
requirements = [f"{package}=={version}" for package, version in settings["packages"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)
for package, version in settings["packages"].items():
    loaded = sys.modules.get(package.replace("-", "_"))
    if loaded is not None and getattr(loaded, "__version__", version) != version:
        raise RuntimeError("Импортирована другая версия библиотеки. Перезапустите Python и выполните ячейки заново.")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Модель", selection["model_id"])
print("Ревизия", selection["revision"])
print("Число шагов", settings["optimizer_steps"])


## 1. Проверяем корпус и сохраняем условия

Состав восьми примеров уже зафиксирован в tiny_config.json.
Validation используется для измерения loss. В обучение поступают восемь
train-примеров. В папку результата сохраняются код, данные, настройки и ошибки.


In [ ]:
import platform
import shutil
import time
import traceback
from datetime import datetime, timezone

import torch

from checks.screening_v3 import read_jsonl, sha256_file, write_json, write_jsonl, finish_reason, technical_indicators
from checks.sft_smoke_v3 import build_masked_example
from checks.sft_tiny_v3 import select_training_records, evaluate_model, train_tiny_model

if not torch.cuda.is_available():
    raise RuntimeError("Выберите Среда выполнения → Сменить среду выполнения → T4 GPU.")
if "tiny_model" in globals():
    raise RuntimeError("Для нового полного запуска завершите текущую GPU-сессию и создайте новую.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
CORPUS_DIR = ROOT / settings["corpus_directory"]
if sha256_file(CORPUS_DIR / "manifest.json") != settings["corpus_manifest_sha256"]:
    raise ValueError("Версия корпуса отличается от зафиксированной для этого запуска")
subprocess.run([sys.executable, str(CORPUS_DIR / "validate.py")], check=True)
corpus_manifest = json.loads((CORPUS_DIR / "manifest.json").read_text(encoding="utf-8"))
records = read_jsonl(CORPUS_DIR / "records.jsonl")
train_rows = select_training_records(records, settings["sample_ids"], selection["system_prompt"])
validation_rows = [row for row in records if row["split"] == "validation"]
if len(validation_rows) != 16:
    raise ValueError("Ожидались все 16 validation-примеров")
if {row["group_id"] for row in train_rows} & {row["group_id"] for row in validation_rows}:
    raise ValueError("Группа сценариев пересекает train и validation")

RUN_ID = "qwen3_4b_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
OUT_DIR = ROOT / "results/sft_v3_tiny" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
ADAPTER_DIR = OUT_DIR / "adapter"
tracked_files = [
    "data/sft_v3/selection.json", "data/sft_v3/tiny_config.json",
    "checks/sft_smoke_v3.py", "checks/sft_tiny_v3.py", "checks/screening_v3.py",
    "checks/autocheck.py", "checks/evaluation.py", "script/prepare_sft_v3.py",
    "notebooks/03_sft_v3_tiny.ipynb", "data/sft_v3/corpus_v1/manifest.json",
    "data/sft_v3/corpus_v1/validate.py",
]
tracked_files.extend(str((CORPUS_DIR / name).relative_to(ROOT)) for name in corpus_manifest["data_file_sha256"])
tracked_files = sorted(set(tracked_files))
metadata = {
    "run_id": RUN_ID, "status": "prepared", "experiment_id": settings["experiment_id"],
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection": selection, "settings": settings,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "file_sha256": {name: sha256_file(ROOT / name) for name in tracked_files},
    "versions": {package: importlib.metadata.version(package) for package in ["torch", *settings["packages"]]},
    "hardware": {
        "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(), "torch_cuda": torch.version.cuda,
        "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3,
        "nvidia_smi": subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True,
        ).strip(),
    },
    "generation_context": "Prepared k-bit base with fresh zero-contribution LoRA before training; greedy generation.",
    "corpus_review_status": corpus_manifest["status"],
}
write_json(OUT_DIR / "run_metadata.json", metadata)
write_jsonl(OUT_DIR / "errors.jsonl", [])
write_jsonl(OUT_DIR / "training_subset.jsonl", train_rows)
write_jsonl(OUT_DIR / "validation_subset.jsonl", validation_rows)
for name in tracked_files:
    destination = OUT_DIR / "reproduction" / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / name, destination)
(OUT_DIR / "pip_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8",
)
training_state = {"started": False, "completed": False}


def record_failure(stage, error):
    errors = read_jsonl(OUT_DIR / "errors.jsonl")
    errors.append({"stage": stage, "type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()})
    write_jsonl(OUT_DIR / "errors.jsonl", errors)
    metadata["status"] = "failed"
    metadata["failed_stage"] = stage
    write_json(OUT_DIR / "run_metadata.json", metadata)


print(json.dumps(metadata["hardware"], ensure_ascii=False, indent=2))
for row in train_rows:
    print(row["sample_id"], row["category"], "dangerous", row["dangerous"])
print("Результаты", OUT_DIR)


## 2. Загружаем модель, строим маски и подключаем LoRA

Используется та же ревизия, NF4, FP16 и eager attention, которые прошли
T4-проверку. Labels=-100 исключает system, user и padding из loss.
Матрицы B свежей LoRA должны быть нулевыми. Сравниваются logits
подготовленной модели с включённым и отключённым адаптером.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

if "tiny_model" in globals():
    raise RuntimeError("Модель уже загружена. Продолжайте со следующей ячейки или начните новый полный запуск.")
set_seed(settings["seed"])
try:
    load_started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(selection["model_id"], revision=selection["revision"], trust_remote_code=False)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    quantization = dict(settings["quantization"])
    quantization["bnb_4bit_compute_dtype"] = torch.float16
    tiny_model = AutoModelForCausalLM.from_pretrained(
        selection["model_id"], revision=selection["revision"], trust_remote_code=False,
        dtype=torch.float16, device_map={"": 0}, quantization_config=BitsAndBytesConfig(**quantization),
        attn_implementation=settings["attention"],
    ).eval()
    original_generation = tiny_model.generation_config.to_dict()
    eos_ids = original_generation["eos_token_id"]
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    if set(eos_ids) != {151645, 151643}:
        raise ValueError("Неожиданные EOS выбранной Qwen-модели")
    if not getattr(tiny_model, "is_loaded_in_4bit", False):
        raise ValueError("Не подтверждена загрузка NF4")
    if any(parameter.device.type != "cuda" for parameter in tiny_model.parameters()):
        raise ValueError("Часть весов оказалась вне GPU")
    generation_config = GenerationConfig(**settings["generation"], eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
    train_pairs = [build_masked_example(tokenizer, row, settings["max_seq_length"]) for row in train_rows]
    validation_pairs = [build_masked_example(tokenizer, row, settings["max_seq_length"]) for row in validation_rows]
    train_batches, train_audit = map(list, zip(*train_pairs))
    validation_batches, validation_audit = map(list, zip(*validation_pairs))
    write_json(OUT_DIR / "mask_audit.json", {"train": train_audit, "validation": validation_audit})

    def prompt_inputs(index):
        ids = train_batches[index]["input_ids"][:train_audit[index]["prompt_tokens"]]
        return {"input_ids": torch.tensor([ids], dtype=torch.long, device="cuda"),
                "attention_mask": torch.ones((1, len(ids)), dtype=torch.long, device="cuda")}

    @torch.inference_mode()
    def collect_logits(indices):
        tiny_model.eval()
        result = {}
        for index in indices:
            inputs = prompt_inputs(index)
            logits = tiny_model(**inputs, use_cache=False, logits_to_keep=1).logits[0, -1].float().cpu()
            if not torch.isfinite(logits).all():
                raise FloatingPointError("Контрольные logits содержат NaN / Inf")
            result[train_rows[index]["sample_id"]] = logits
        return result

    original_logits = collect_logits([0])
    tiny_model = prepare_model_for_kbit_training(
        tiny_model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    tiny_model.config.use_cache = False
    prepared_logits = collect_logits([0])
    tiny_model = get_peft_model(tiny_model, LoraConfig(**settings["lora"]))
    active_logits = collect_logits([0])
    with tiny_model.disable_adapter():
        fresh_disabled_logits = collect_logits([0])
    probe_id = train_rows[0]["sample_id"]
    b_parameters = [parameter for name, parameter in tiny_model.named_parameters() if "lora_B" in name]
    identity = {
        "B_matrices": len(b_parameters),
        "all_B_zero": bool(b_parameters) and all(torch.count_nonzero(parameter).item() == 0 for parameter in b_parameters),
        "active_vs_disabled_max_abs_diff": float((active_logits[probe_id] - fresh_disabled_logits[probe_id]).abs().max()),
        "active_vs_disabled_allclose": torch.allclose(active_logits[probe_id], fresh_disabled_logits[probe_id], rtol=1e-5, atol=1e-5),
        "preparation_max_abs_diff": float((original_logits[probe_id] - prepared_logits[probe_id]).abs().max()),
        "base_weights_frozen": all(not parameter.requires_grad for name, parameter in tiny_model.named_parameters() if "lora_" not in name),
        "gradient_checkpointing": tiny_model.is_gradient_checkpointing,
        "trainable_parameters": sum(parameter.numel() for parameter in tiny_model.parameters() if parameter.requires_grad),
    }
    write_json(OUT_DIR / "fresh_adapter_check.json", identity)
    if not all(identity[key] for key in ["all_B_zero", "active_vs_disabled_allclose", "base_weights_frozen", "gradient_checkpointing"]):
        raise ValueError("Не прошла проверка свежей LoRA")
    del b_parameters
    metadata["model_load_seconds"] = time.perf_counter() - load_started
    metadata["status"] = "fresh_adapter_verified"
    metadata["generation_config"] = generation_config.to_dict()
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("load_masks_and_fresh_adapter", error)
    raise

print(json.dumps(identity, ensure_ascii=False, indent=2))


## 3. Измеряем исходный loss и сохраняем восемь ответов

Loss и точность токенов вычисляются при поданном эталонном продолжении,
что позволяет измерять соответствие обучающим targets. Отдельно модель
генерирует ответ, получая только system и user. Ответы сохраняются
после каждого вопроса, включая случаи достижения лимита.


In [ ]:
def save_evaluation(step, include_validation):
    evaluations = read_jsonl(OUT_DIR / "evaluations.jsonl")
    splits = [("train", train_batches, train_rows)]
    if include_validation:
        splits.append(("validation", validation_batches, validation_rows))
    for split, batches, rows in splits:
        result = evaluate_model(tiny_model, batches, [row["sample_id"] for row in rows], "cuda", torch.float16)
        entry = {"step": step, "split": split, **result}
        evaluations = [row for row in evaluations if (row["step"], row["split"]) != (step, split)]
        evaluations.append(entry)
        write_jsonl(OUT_DIR / "evaluations.jsonl", evaluations)
        print("step", step, split, "loss", round(result["token_weighted_loss"], 4),
              "token accuracy", round(result["token_accuracy"], 4), "EOS accuracy", round(result["eos_accuracy"], 4))


@torch.inference_mode()
def generate_phase(phase):
    path = OUT_DIR / ("answers_" + phase + ".jsonl")
    answers = read_jsonl(path)
    index = {row["sample_id"]: row for row in answers}
    expected = {row["sample_id"] for row in train_rows}
    if len(index) != len(answers) or not set(index) <= expected:
        raise ValueError("Неожиданные или повторные ID сохранённых ответов")
    tiny_model.eval()
    for position, row in enumerate(train_rows):
        sample_id = row["sample_id"]
        if sample_id in index:
            if index[sample_id]["phase"] != phase or index[sample_id]["question"] != row["messages"][1]["content"]:
                raise ValueError("Сохранённый ответ относится к другому вопросу или этапу")
            continue
        inputs = prompt_inputs(position)
        started = time.perf_counter()
        output = tiny_model.generate(**inputs, generation_config=generation_config)
        token_ids = output[0, inputs["input_ids"].shape[1]:].tolist()
        answer = tokenizer.decode(token_ids, skip_special_tokens=True)
        entry = {
            "sample_id": sample_id, "category": row["category"], "dangerous": row["dangerous"], "phase": phase,
            "question": row["messages"][1]["content"], "answer": answer,
            "generated_token_ids": token_ids, "generated_tokens": len(token_ids),
            "finish_reason": finish_reason(token_ids, eos_ids, settings["generation"]["max_new_tokens"]),
            "max_new_tokens": settings["generation"]["max_new_tokens"], "eos_token_ids": eos_ids,
            "elapsed_seconds": time.perf_counter() - started,
            "technical_indicators": technical_indicators(row["messages"][1]["content"], answer),
        }
        index[sample_id] = entry
        answers = [index[item["sample_id"]] for item in train_rows if item["sample_id"] in index]
        write_jsonl(path, answers)
        print(phase, sample_id, "tokens", entry["generated_tokens"], entry["finish_reason"])
        print(answer[:400])
        del inputs, output
    return answers


if training_state["started"]:
    raise RuntimeError("Обучение уже начиналось. Исходные показатели сохранены в evaluations.jsonl.")
try:
    save_evaluation(0, include_validation=True)
    answers_before = generate_phase("before")
    metadata["status"] = "before_training_saved"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("before_training", error)
    raise


## 4. Выполняем 24 шага обучения

Градиенты накапливаются по четырём примерам, затем обновляются LoRA-веса.
Вклад каждого примера взвешен по числу обучаемых токенов.
Loss и градиенты проверяются на NaN / Inf. Train оценивается на шагах
4, 12 и 24, validation дополнительно оценивается в конце.

После ошибки в этой ячейке сохраните ZIP и начните новый полный запуск
в новой сессии. Журнал содержит завершённые шаги и порядок примеров.


In [ ]:
if training_state["started"]:
    raise RuntimeError("Обучение уже запускалось. При успехе продолжайте к ответам после обучения, при ошибке сохраните ZIP.")
if metadata["status"] != "before_training_saved":
    raise RuntimeError("Сначала завершите сохранение исходных ответов и loss")
training_state["started"] = True
training_history = []


def on_step(entry):
    training_history.append(entry)
    write_jsonl(OUT_DIR / "training_steps.jsonl", training_history)
    metadata["last_optimizer_step"] = entry["step"]
    write_json(OUT_DIR / "run_metadata.json", metadata)
    print("step", entry["step"], "loss", round(entry["token_weighted_loss"], 4),
          "grad norm", round(entry["gradient_norm_before_clip"], 4))


try:
    metadata["status"] = "training"
    write_json(OUT_DIR / "run_metadata.json", metadata)
    torch.cuda.empty_cache()
    training_summary = train_tiny_model(
        tiny_model, train_batches, [row["sample_id"] for row in train_rows], settings,
        "cuda", torch.float16, on_step,
        lambda step: save_evaluation(step, include_validation=step == settings["optimizer_steps"]),
    )
    write_json(OUT_DIR / "training_summary.json", training_summary)
    if training_summary["optimizer_steps"] != settings["optimizer_steps"] or not training_summary["nonzero_B_matrices"]:
        raise ValueError("Обновление LoRA не подтверждено")
    training_state["completed"] = True
    metadata["status"] = "training_complete"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("training", error)
    raise

print(json.dumps(training_summary, ensure_ascii=False, indent=2))


## 5. Сохраняем ответы после обучения и проверяем вклад адаптера

Те же восемь вопросов генерируются при том же greedy-режиме.
Дополнительно проверяются изменение logits при отключении обученной LoRA
и совпадение контрольного результата отключённой базы с её исходным состоянием.


In [ ]:
if not training_state["completed"]:
    raise RuntimeError("Обучение ещё не завершено")
try:
    answers_after = generate_phase("after")
    trained_logits = collect_logits(range(len(train_rows)))
    with tiny_model.disable_adapter():
        trained_disabled_logits = collect_logits(range(len(train_rows)))
    influence = {
        "trained_adapter_max_abs_diff": max(float((trained_logits[key] - trained_disabled_logits[key]).abs().max()) for key in trained_logits),
        "disabled_base_probe_max_abs_diff": float((trained_disabled_logits[probe_id] - fresh_disabled_logits[probe_id]).abs().max()),
        "disabled_base_probe_allclose": torch.allclose(trained_disabled_logits[probe_id], fresh_disabled_logits[probe_id], rtol=1e-5, atol=1e-5),
    }
    write_json(OUT_DIR / "trained_adapter_check.json", influence)
    if influence["trained_adapter_max_abs_diff"] == 0 or not influence["disabled_base_probe_allclose"]:
        raise ValueError("Не прошла проверка вклада обученного адаптера или отключённой базы")
    metadata["status"] = "after_training_saved"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("after_training", error)
    raise

print(json.dumps(influence, ensure_ascii=False, indent=2))


## 6. Сохраняем адаптер и сравниваем после загрузки

Адаптер сохраняется в safetensors. Все тензоры проверяются на конечность.
Затем LoRA выгружается из базовой модели и подключается из сохранённой папки.
Для всех восьми вопросов сравниваются logits и полные greedy-последовательности.


In [ ]:
from safetensors import safe_open

try:
    if not training_state["completed"] or len(read_jsonl(OUT_DIR / "answers_after.jsonl")) != 8:
        raise RuntimeError("Сначала завершите обучение и восемь ответов после него")
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    tiny_model.peft_config["default"].base_model_name_or_path = selection["model_id"]
    tiny_model.peft_config["default"].revision = selection["revision"]
    tiny_model.save_pretrained(ADAPTER_DIR, safe_serialization=True, save_embedding_layers=False)
    tokenizer.save_pretrained(ADAPTER_DIR)
    weights_path = ADAPTER_DIR / "adapter_model.safetensors"
    tensor_count = 0
    with safe_open(weights_path, framework="pt", device="cpu") as saved:
        for key in saved.keys():
            if not torch.isfinite(saved.get_tensor(key)).all():
                raise FloatingPointError("Сохранённые веса содержат NaN / Inf")
            tensor_count += 1
    base_for_reload = tiny_model.unload()
    del tiny_model
    tiny_model = PeftModel.from_pretrained(base_for_reload, ADAPTER_DIR, is_trainable=False).eval()
    tiny_model.config.use_cache = False
    reloaded_logits = collect_logits(range(len(train_rows)))
    answers_after_reload = generate_phase("after_reload")
    reloaded_index = {row["sample_id"]: row for row in answers_after_reload}
    per_example = [{
        "sample_id": row["sample_id"],
        "logits_max_abs_diff": float((trained_logits[row["sample_id"]] - reloaded_logits[row["sample_id"]]).abs().max()),
        "logits_allclose": torch.allclose(trained_logits[row["sample_id"]], reloaded_logits[row["sample_id"]], rtol=1e-4, atol=1e-4),
        "greedy_tokens_equal": row["generated_token_ids"] == reloaded_index[row["sample_id"]]["generated_token_ids"],
    } for row in answers_after]
    roundtrip = {"adapter_sha256": sha256_file(weights_path), "adapter_bytes": weights_path.stat().st_size,
                 "saved_tensors": tensor_count, "per_example": per_example,
                 "all_logits_match": all(row["logits_allclose"] for row in per_example),
                 "all_greedy_tokens_match": all(row["greedy_tokens_equal"] for row in per_example)}
    write_json(OUT_DIR / "save_reload_check.json", roundtrip)
    if not roundtrip["all_logits_match"] or not roundtrip["all_greedy_tokens_match"]:
        raise ValueError("Контрольные результаты после загрузки адаптера отличаются")
    metadata["status"] = "reload_verified"
    metadata["adapter_sha256"] = roundtrip["adapter_sha256"]
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("save_and_reload", error)
    raise

print("Загрузка проверена на", len(per_example), "вопросах")
print("SHA-256 адаптера", roundtrip["adapter_sha256"])


## 7. Формируем итог и сравнение ответов

`TINY PIPELINE PASS` означает прохождение технических проверок.
Изменение loss и точности токенов сохраняется численно, включая возможное
ухудшение. В comparison.md находятся вопрос, эталон, критерии и два ответа.
Эти тексты нужно учитывать при выводе об усвоении примеров.


In [ ]:
try:
    if metadata["status"] not in {"reload_verified", "complete"}:
        raise RuntimeError("Сначала завершите проверку сохранения и загрузки")
    evaluations = read_jsonl(OUT_DIR / "evaluations.jsonl")
    evaluation_index = {(row["step"], row["split"]): row for row in evaluations}
    initial_train = evaluation_index[(0, "train")]
    final_train = evaluation_index[(settings["optimizer_steps"], "train")]
    initial_validation = evaluation_index[(0, "validation")]
    final_validation = evaluation_index[(settings["optimizer_steps"], "validation")]
    fit_summary = {
        "train_loss_before": initial_train["token_weighted_loss"], "train_loss_after": final_train["token_weighted_loss"],
        "train_loss_relative_drop": (1 - final_train["token_weighted_loss"] / initial_train["token_weighted_loss"]) if initial_train["token_weighted_loss"] else None,
        "train_loss_decreased": final_train["token_weighted_loss"] < initial_train["token_weighted_loss"],
        "train_token_accuracy_before": initial_train["token_accuracy"], "train_token_accuracy_after": final_train["token_accuracy"],
        "train_eos_accuracy_before": initial_train["eos_accuracy"], "train_eos_accuracy_after": final_train["eos_accuracy"],
        "validation_loss_before": initial_validation["token_weighted_loss"], "validation_loss_after": final_validation["token_weighted_loss"],
        "generation": {phase: {"n": len(rows), "length_stops": sum(row["finish_reason"] == "length" for row in rows),
                               "empty_answers": sum(not row["answer"].strip() for row in rows)}
                       for phase, rows in [("before", answers_before), ("after", answers_after), ("after_reload", answers_after_reload)]},
        "content_review": "pending review of comparison.md against the saved criteria",
    }
    write_json(OUT_DIR / "fit_summary.json", fit_summary)
    before_index = {row["sample_id"]: row for row in answers_before}
    after_index = {row["sample_id"]: row for row in answers_after}
    report = ["# Обучение на восьми примерах\n\n", f"Прогон `{RUN_ID}`.\n\n",
              "Оценка выполняется на примерах, использованных для обучения.\n\n"]
    for row in train_rows:
        sample_id = row["sample_id"]
        report.append(f"## {sample_id} · {row['category']}\n\n" + row["messages"][1]["content"] + "\n\n")
        report.append("### Эталон\n\n" + row["messages"][2]["content"] + "\n\n")
        for criterion in row["rubric"]["required_criteria"]:
            report.append("- " + criterion["id"] + " — " + criterion["text"] + "\n")
        for title, answer in [("До обучения", before_index[sample_id]), ("После обучения", after_index[sample_id])]:
            report.append("\n### " + title + "\n\n" + answer["answer"] + "\n\n")
            report.append(f"Остановка `{answer['finish_reason']}`, токенов {answer['generated_tokens']}.\n\n")
    (OUT_DIR / "comparison.md").write_text("".join(report).rstrip() + "\n", encoding="utf-8")
    checks = {"status": "PASS", "train_examples": 8, "validation_examples": 16,
              "fresh_adapter_identity": identity["active_vs_disabled_allclose"],
              "base_weights_frozen": identity["base_weights_frozen"], "optimizer_steps": training_summary["optimizer_steps"],
              "trained_adapter_changes_logits": influence["trained_adapter_max_abs_diff"] > 0,
              "disabled_base_probe_matches": influence["disabled_base_probe_allclose"],
              "saved_weights_finite": True, "reload_logits_match": roundtrip["all_logits_match"],
              "reload_greedy_tokens_match": roundtrip["all_greedy_tokens_match"],
              "train_loss_decreased": fit_summary["train_loss_decreased"],
              "logged_errors": len(read_jsonl(OUT_DIR / "errors.jsonl"))}
    write_json(OUT_DIR / "checks.json", checks)
    metadata["status"] = "complete"
    metadata.pop("failed_stage", None)
    metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("summary", error)
    raise

print("TINY PIPELINE PASS")
print(json.dumps(fit_summary, ensure_ascii=False, indent=2))
print("Пик выделенной памяти, GiB", training_summary["peak_allocated_gib"])


## 8. Скачиваем результаты

Последняя ячейка скачает ZIP через браузер. Её можно выполнить после ошибки,
если OUT_DIR уже создан. Архив содержит метрики, тексты, журнал, снимок кода
и папку adapter с весами safetensors.

Сохраните выполненный ноутбук через **Файл → Скачать → Скачать файл IPYNB**.
Для разбора результата нужны ZIP и IPYNB одного запуска.
После проверки результатов следующая работа — SFT-пилот на 80 train-примерах.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    str(Path("/content") / ("sft_v3_tiny_" + RUN_ID)),
    "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name,
)
files.download(archive)
print("Скачивается", archive)
print("Run ID", RUN_ID)
print("Статус", json.loads((OUT_DIR / "run_metadata.json").read_text(encoding="utf-8"))["status"])
